# EVolvAI — Physics-Informed Generative Counterfactual VAE
## Research Training Notebook | IEEE Submission Build

---

### Architecture Summary
This notebook implements the complete, end-to-end training pipeline for the **Generative Counterfactual Demand VAE (GCD-VAE)**:
- **Encoder**: Causal Temporal Convolutional Network (TCN) → Multi-Head Self-Attention → Gaussian Posterior
- **Decoder**: Condition-concatenated (Z ⊕ C) → FC projection → TCN decoder
- **Physics Engine**: Differentiable LinDistFlow (V² form) on the IEEE 33-Bus radial feeder — voltage, thermal, and transformer penalties are backpropagated directly through the generator
- **Data**: NYC PlugNYC EV Charging Sessions (2021–2026), NYC ATVC Hourly Traffic Volumes (2021–2026), Open-Meteo NYC Weather, bootstrapped to **5,000 synthetic daily scenarios × 32 nodes**

### Training Strategy (2-Phase)
| Phase | Epochs | KLD Weight | Physics λ | Purpose |
|---|---|---|---|---|
| Warm-Up | 1-50 | 0 → 0.01 | 0 | Reconstruction focus, avoid posterior collapse |
| Physics-On | 51-200 | 0.01 → 0.10 | Activated | LinDistFlow constraints shape the latent manifold |


## §1 · Environment Setup & Repository Clone

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 2>/dev/null || \
!pip install -q torch torchvision  # CPU fallback
!pip install -q pandas numpy matplotlib scikit-learn pyarrow tqdm openmeteo-requests requests-cache retry-requests

import sys, os

# ── Colab: mount Drive and clone repo ────────────────────────────────────────
IS_COLAB = False
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IS_COLAB  = True
    DRIVE_DIR = '/content/drive/MyDrive/EVolvAI_Research'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"✅  Running on Colab | Drive mounted at: {DRIVE_DIR}")
except ImportError:
    DRIVE_DIR = None
    print("✅  Running locally")

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO_ROOT = '/content/EVolvAI' if IS_COLAB else os.getcwd()
if IS_COLAB and not os.path.exists(REPO_ROOT):
    !git clone https://github.com/seeramsujay/EVolvAI.git $REPO_ROOT
if IS_COLAB:
    %cd $REPO_ROOT

sys.path.insert(0, REPO_ROOT)
print(f"   CWD = {os.getcwd()}")
print(f"   IS_COLAB   = {IS_COLAB}")
print(f"   DRIVE_DIR  = {DRIVE_DIR}")


## §2 · Raw Data Acquisition
All raw files are git-ignored (binary/large). This cell downloads them if missing.

| Dataset | Source | Size | Purpose |
|---|---|---|---|
| `nyc_charging.csv` | NYC PlugNYC Open Data | ~43 MB / 233k rows | Primary EV charging session logs (2021–2026) |
| `Automated_Traffic_Volume_Counts_20260415.csv` | NYC Open Data ATVC | ~70 MB / 436k rows | Hourly traffic volume by segment (2021–2026) |


In [ ]:
import os, subprocess

def download_if_missing(url, dest, description=""):
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f"  ✅  {description}: already present ({size_mb:.1f} MB)")
        return
    print(f"  ⬇️   Downloading {description}...")
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    result = subprocess.run(['curl', '-L', '-o', dest, url],
                            capture_output=True, text=True)
    if result.returncode == 0:
        size_mb = os.path.getsize(dest) / 1e6
        print(f"  ✅  {description}: downloaded ({size_mb:.1f} MB)")
    else:
        print(f"  ❌  {description}: download failed\n{result.stderr[:300]}")

print("\n📥  Checking raw data dependencies...")

# NYC PlugNYC EV Charging Sessions
download_if_missing(
    url  = "https://data.cityofnewyork.us/api/views/kj7g-u4gp/rows.csv?accessType=DOWNLOAD",
    dest = "data/raw/nyc_charging.csv",
    description = "NYC PlugNYC EV Charging Sessions (2021–2026)"
)

# NYC Automated Traffic Volume Counts (ATVC)
# Columns: RequestID, Boro, Yr, M, D, HH, MM, Vol, SegmentID, WktGeom, street, fromSt, toSt, Direction
download_if_missing(
    url  = "https://data.cityofnewyork.us/api/views/7ym2-wayt/rows.csv?accessType=DOWNLOAD",
    dest = "data/raw/Automated_Traffic_Volume_Counts_20260415.csv",
    description = "NYC ATVC Hourly Traffic Volumes (2021–2026)"
)

print("\n📦  Data directory contents:")
for root, dirs, files in os.walk("data"):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    for f in files:
        full = os.path.join(root, f)
        print(f"  {full}  ({os.path.getsize(full)/1e6:.2f} MB)")


## §3 · Weather Data Integration (NYC — Open-Meteo)
Fetches **hourly** weather for **New York City (lat=40.7128, lon=-74.0060)** spanning **2021-07-31 → 2026-04-13** to perfectly bound the EV charging dataset.

Uses the Open-Meteo Historical API (no key required).

Features: `temperature_2m`, `precipitation`, `cloudcover` → used to derive `solar_availability`.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

WEATHER_OUT = "data/processed/weather_features.parquet"

# NYC bounding dates (matched to the EV charging dataset)
NYC_LAT        = 40.7128
NYC_LON        = -74.0060
WEATHER_START  = "2021-07-31"
WEATHER_END    = "2026-04-13"

if os.path.exists(WEATHER_OUT):
    df_weather = pd.read_parquet(WEATHER_OUT)
    print(f"✅  Weather cache found: {df_weather.shape[0]:,} hourly records")
else:
    print("📡  Fetching NYC hourly weather from Open-Meteo Historical API...")
    print(f"    Location : New York City ({NYC_LAT}, {NYC_LON})")
    print(f"    Range    : {WEATHER_START} → {WEATHER_END}")

    try:
        import openmeteo_requests
        import requests_cache
        from retry_requests import retry

        # Setup cached session to avoid hammering the API on re-runs
        cache_session = requests_cache.CachedSession('.openmeteo_cache', expire_after=-1)
        retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
        client = openmeteo_requests.Client(session=retry_session)

        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude"        : NYC_LAT,
            "longitude"       : NYC_LON,
            "start_date"      : WEATHER_START,
            "end_date"        : WEATHER_END,
            "hourly"          : ["temperature_2m", "precipitation", "cloudcover"],
            "timezone"        : "America/New_York",
            "wind_speed_unit" : "mph"
        }

        responses = client.weather_api(url, params=params)
        resp      = responses[0]
        hourly    = resp.Hourly()

        timestamps     = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert("America/New_York"),
            periods=hourly.Variables(0).ValuesAsNumpy().shape[0],
            freq="h"
        ).tz_localize(None)  # drop tz for parquet compat

        temp_arr   = hourly.Variables(0).ValuesAsNumpy()   # temperature_2m [°C]
        precip_arr = hourly.Variables(1).ValuesAsNumpy()   # precipitation [mm]
        cloud_arr  = hourly.Variables(2).ValuesAsNumpy()   # cloudcover [0-100%]

        df_weather = pd.DataFrame({
            'timestamp'          : timestamps,
            'temperature_c'      : temp_arr,
            'precipitation_mm'   : precip_arr,
            'solar_availability' : np.clip(1.0 - cloud_arr / 100.0, 0.0, 1.0),
        })

        df_weather = df_weather.dropna()
        df_weather['date'] = df_weather['timestamp'].dt.strftime('%Y-%m-%d')
        df_weather['hour'] = df_weather['timestamp'].dt.hour
        df_weather = df_weather[['date', 'hour', 'temperature_c', 'precipitation_mm', 'solar_availability']]

        os.makedirs("data/processed", exist_ok=True)
        df_weather.to_parquet(WEATHER_OUT, index=False)
        print(f"✅  NYC weather fetched & cached: {df_weather.shape}")

    except Exception as e:
        print(f"⚠️   Open-Meteo failed ({e}) — building synthetic NYC placeholder...")
        # Synthetic NYC approximation: cold winters, warm summers, bimodal seasons
        dates = pd.date_range(WEATHER_START, WEATHER_END, freq='D')
        rows = []
        for d in dates:
            doy = d.timetuple().tm_yday
            for h in range(24):
                # NYC avg temp: ~2°C winter peak, ~27°C summer peak
                temp_seasonal = 14.5 + 12.5 * np.sin((doy - 80) * 2 * np.pi / 365)
                temp = temp_seasonal + 3 * np.sin(h * np.pi / 12)
                solar = max(0.0, np.sin((h - 6) * np.pi / 12))
                rows.append({'date': d.strftime('%Y-%m-%d'), 'hour': h,
                             'temperature_c': float(temp),
                             'precipitation_mm': 0.0,
                             'solar_availability': float(solar)})
        df_weather = pd.DataFrame(rows)
        os.makedirs("data/processed", exist_ok=True)
        df_weather.to_parquet(WEATHER_OUT, index=False)
        print(f"✅  Synthetic NYC weather generated: {df_weather.shape}")

print(f"\n  Weather date range  : {df_weather['date'].min()} → {df_weather['date'].max()}")
print(f"  Records             : {len(df_weather):,}")
print(df_weather.describe().round(3))


## §4 · NYC ATVC Traffic Index Preprocessing
Converts the **NYC Automated Traffic Volume Counts** dataset into a per-hour scalar `traffic_index ∈ [0,1]`.

**Schema**: `Yr, M, D, HH, MM, Vol` — vehicle counts at 15-minute intervals across all NYC boroughs (2021–2026, 436k rows).

Aggregation: sum `Vol` across all segments per `(Yr, M, D, HH)` → normalise globally → average across all days → canonical 24-hr NYC traffic profile.

This **causal exogenous variable** tells the attention mechanism *when* vehicles arrive at charging nodes.


In [ ]:
import numpy as np
import pandas as pd

ATVC_PATH    = "data/raw/Automated_Traffic_Volume_Counts_20260415.csv"
TRAFFIC_OUT  = "data/processed/traffic_tensor_24h.npy"
TRAFFIC_FULL = "data/processed/traffic_daily_24h.parquet"  # per-day profiles for dataset join

if os.path.exists(TRAFFIC_OUT):
    traffic_24h = np.load(TRAFFIC_OUT)
    print(f"✅  Traffic tensor cached: {traffic_24h.shape}")
else:
    print("🚦  Building NYC hourly traffic index from ATVC data...")
    print(f"    Source: {ATVC_PATH}")

    try:
        # Columns: RequestID, Boro, Yr, M, D, HH, MM, Vol, SegmentID, ...
        # HH = hour (0-23), MM = minute (0, 15, 30, 45)
        # Vol = vehicle count per 15-min interval at that segment
        usecols = ['Yr', 'M', 'D', 'HH', 'Vol']
        atvc = pd.read_csv(ATVC_PATH, usecols=usecols, dtype={'Yr': int, 'M': int, 'D': int, 'HH': int, 'Vol': int})
        print(f"  Loaded {len(atvc):,} rows | Yr range: {atvc['Yr'].min()}–{atvc['Yr'].max()}")

        # Build date string for per-day join
        atvc['date'] = (atvc['Yr'].astype(str) + '-' +
                        atvc['M'].astype(str).str.zfill(2) + '-' +
                        atvc['D'].astype(str).str.zfill(2))

        # Aggregate: sum all segment volumes per (date, hour)
        hourly = atvc.groupby(['date', 'HH'])['Vol'].sum().reset_index()
        hourly.columns = ['date', 'hour', 'total_vol']

        # Pivot to (n_days, 24)
        pivot = hourly.pivot_table(index='date', columns='hour', values='total_vol', fill_value=0)
        pivot = pivot.reindex(columns=range(24), fill_value=0)
        daily_vol = pivot.values.astype(np.float32)  # (n_days, 24)

        # Global normalise to [0, 1]
        lo, hi = daily_vol.min(), daily_vol.max()
        daily_norm = (daily_vol - lo) / (hi - lo + 1e-8)

        # Save full per-day profiles (used in dataset builder for per-day traffic)
        daily_df = pd.DataFrame(daily_norm, index=pivot.index, columns=[f"h{h:02d}" for h in range(24)])
        daily_df.index.name = 'date'
        os.makedirs("data/processed", exist_ok=True)
        daily_df.reset_index().to_parquet(TRAFFIC_FULL, index=False)
        print(f"  ✅  Per-day traffic profiles saved: {daily_df.shape} → {TRAFFIC_FULL}")

        # Canonical 24-hr profile: mean across all days
        traffic_24h = daily_norm.mean(axis=0).astype(np.float32)  # (24,)
        np.save(TRAFFIC_OUT, traffic_24h)
        print(f"✅  NYC ATVC traffic profile built: {traffic_24h.shape}  range=[{traffic_24h.min():.3f}, {traffic_24h.max():.3f}]")

    except Exception as e:
        print(f"⚠️   ATVC parse failed ({e}). Building FHWA synthetic fallback (NYC-tuned)...")
        # NYC-specific double-hump: pronounced morning rush (8am) + evening (6pm), never fully quiet
        fhwa_nyc = np.array([0.018, 0.013, 0.010, 0.009, 0.011, 0.020,
                              0.040, 0.078, 0.095, 0.075, 0.060, 0.055,
                              0.058, 0.055, 0.058, 0.070, 0.088, 0.100,
                              0.090, 0.068, 0.048, 0.036, 0.027, 0.022], dtype=np.float32)
        traffic_24h = (fhwa_nyc - fhwa_nyc.min()) / (fhwa_nyc.max() - fhwa_nyc.min())
        np.save(TRAFFIC_OUT, traffic_24h)
        print(f"✅  NYC-tuned FHWA synthetic profile: {traffic_24h.shape}")

# Pretty print
print("\n  Hourly NYC Traffic Index Profile:")
print("  " + "─"*50)
for h in range(24):
    bar = "█" * int(traffic_24h[h] * 35)
    print(f"  {h:02d}:00  {traffic_24h[h]:.3f}  {bar}")


## §5 · High-Performance Bootstrap (NYC-Grounded)
Groups the **233,865 NYC charging sessions** (2021–2026) **by historical day**, then bootstraps **5,000 synthetic days** by sampling historical days with replacement and distributing individual sessions across the 32 IEEE-33 bus load nodes probabilistically, weighted by `traffic_index[hour]`.

**NYC charging schema**: `Date, Connected Time, Disconnected Time, Energy Provided (kWh), Charge Duration (min)`

The key insight: instead of genetic extrapolation, we resample real NYC charging behaviour and let the actual hourly NYC traffic flow determine *which node absorbs each session*. This prevents KL collapse in the VAE decoder.


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

TRAIN_PARQUET  = "data/processed/train_data.parquet"
NUM_NODES      = 32
NUM_SCENARIOS  = 5000

if os.path.exists(TRAIN_PARQUET):
    df_check = pd.read_parquet(TRAIN_PARQUET)
    n_days = df_check['date'].nunique()
    print(f"✅  Bootstrap parquet already exists: {n_days} days × {df_check['node_id'].nunique()} nodes × 24 hrs = {len(df_check):,} rows")
    del df_check
else:
    print("⚙️   [1/5] Loading NYC PlugNYC sessions...")
    df = pd.read_csv("data/raw/nyc_charging.csv")
    print(f"  Raw rows: {len(df):,}")
    print(f"  Columns : {df.columns.tolist()}")

    # ── NYC PlugNYC schema ────────────────────────────────────────────────────
    # Date                  : e.g. '08/21/2025'
    # Connected Time        : e.g. '19:10:54.0000000'
    # Disconnected Time     : e.g. '07:11:12.0000000'
    # Charge Duration (min) : numeric
    # Energy Provided (kWh) : numeric
    # ─────────────────────────────────────────────────────────────────────────
    df['start_dt'] = pd.to_datetime(
        df['Date'].astype(str) + ' ' + df['Connected Time'].astype(str).str[:8],
        format='%m/%d/%Y %H:%M:%S', errors='coerce'
    )
    df['end_dt']   = pd.to_datetime(
        df['Date'].astype(str) + ' ' + df['Disconnected Time'].astype(str).str[:8],
        format='%m/%d/%Y %H:%M:%S', errors='coerce'
    )
    df['kWh']      = pd.to_numeric(df['Energy Provided (kWh)'], errors='coerce')
    df['dur_min']  = pd.to_numeric(df['Charge Duration (min)'], errors='coerce')

    df = df.dropna(subset=['start_dt', 'kWh', 'dur_min'])
    df = df[df['kWh'] > 0]
    df = df[df['dur_min'] > 0]

    # Cross-midnight correction: if end < start, end is next day
    mask = df['end_dt'] < df['start_dt']
    df.loc[mask, 'end_dt'] += pd.Timedelta(days=1)

    df['date']       = df['start_dt'].dt.date
    df['hour']       = df['start_dt'].dt.hour
    df['duration_h'] = (df['dur_min'] / 60.0).clip(lower=0.083)  # min 5 min
    df['avg_kw']     = (df['kWh'] / df['duration_h']).clip(upper=350.0)  # cap at 350 kW (DC fast)

    print(f"  ✅  [2/5] Cleaned sessions: {len(df):,} | Date range: {df['date'].min()} → {df['date'].max()}")

    # Group by historical day
    daily_groups    = dict(tuple(df.groupby('date')))
    historical_days = list(daily_groups.keys())
    print(f"  ✅  [3/5] Historical days available: {len(historical_days)}")

    # Traffic-based node probability matrix (24 × NUM_NODES)
    traffic_24h = np.load("data/processed/traffic_tensor_24h.npy")
    hour_node_probs = np.zeros((24, NUM_NODES), dtype=np.float32)
    for h in range(24):
        w = traffic_24h[h]
        hour_node_probs[h] = np.full(NUM_NODES, w / NUM_NODES)  # uniform across nodes, scaled by hour traffic
    # Normalise each row so it sums to 1
    hour_node_probs = hour_node_probs / hour_node_probs.sum(axis=1, keepdims=True)

    print(f"  ✅  [4/5] Node probability matrix built: {hour_node_probs.shape}")
    print(f"  ⚙️   [5/5] Bootstrapping {NUM_SCENARIOS:,} daily scenarios (this takes ~3-5 min)...")

    rng          = np.random.default_rng(42)
    sampled_idxs = rng.integers(0, len(historical_days), size=NUM_SCENARIOS)
    sampled_days = [historical_days[i] for i in sampled_idxs]
    nodes_arr    = np.arange(NUM_NODES)

    base_date = pd.Timestamp("2021-07-31")

    # Pre-allocate: 5000 × 24 × 32 = 3,840,000 rows
    TOTAL = NUM_SCENARIOS * 24 * NUM_NODES
    out_dates   = np.empty(TOTAL, dtype='U10')
    out_hours   = np.empty(TOTAL, dtype=np.int8)
    out_nodes   = np.empty(TOTAL, dtype=np.int8)
    out_kw      = np.zeros(TOTAL,  dtype=np.float32)

    ptr = 0
    for i in tqdm(range(NUM_SCENARIOS), desc="Bootstrap scenarios"):
        sessions    = daily_groups[sampled_days[i]]
        gen_date_str = (base_date + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
        day_demand  = np.zeros((24, NUM_NODES), dtype=np.float32)

        for _, row in sessions.iterrows():
            sh   = int(row['hour'])
            dur  = max(1, int(round(row['duration_h'])))
            kw   = float(row['avg_kw'])
            node = rng.choice(nodes_arr, p=hour_node_probs[sh])
            for offset in range(dur):
                h = (sh + offset) % 24
                day_demand[h, node] += kw

        block_size = 24 * NUM_NODES
        out_dates[ptr:ptr+block_size]  = gen_date_str
        out_hours[ptr:ptr+block_size]  = np.repeat(np.arange(24), NUM_NODES)
        out_nodes[ptr:ptr+block_size]  = np.tile(nodes_arr, 24)
        out_kw[ptr:ptr+block_size]     = day_demand.flatten()
        ptr += block_size

    final_df = pd.DataFrame({
        'date':      out_dates,
        'hour':      out_hours,
        'node_id':   [f"node_{n:02d}" for n in out_nodes],
        'demand_kw': out_kw,
    })

    os.makedirs("data/processed", exist_ok=True)
    final_df.to_parquet(TRAIN_PARQUET, index=False)
    print(f"\n✅  Bootstrap complete!")
    print(f"   Parquet: {TRAIN_PARQUET}")
    print(f"   Shape  : {final_df.shape}")
    print(f"   kW range: [{final_df['demand_kw'].min():.2f}, {final_df['demand_kw'].max():.2f}]")


## §6 · Hyperparameter Configuration & Hardware Detection
All constants live in a single `CFG` dataclass — changing a value here propagates everywhere.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import r2_score, mean_absolute_error
import math, os, time, json, warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans', 'axes.grid': True})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️   Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"    GPU   : {torch.cuda.get_device_name(0)}")
    print(f"    VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

class CFG:
    # ── Geography ─────────────────────────────────────────────────────────────
    CITY            = "New York City, NY"
    LAT             = 40.7128
    LON             = -74.0060
    DATA_START      = "2021-07-31"   # NYC EV charging dataset start
    DATA_END        = "2026-04-13"   # NYSE EV charging dataset end

    # ── Grid ──────────────────────────────────────────────────────────────────
    NUM_NODES           = 32
    SEQ_LEN             = 24
    NUM_WEATHER_FEAT    = 4       # temp, precip, solar, traffic
    NUM_FEATURES        = NUM_NODES + NUM_WEATHER_FEAT   # 36

    # Condition vector C (length must match COND_DIM)
    # [temp_anomaly, ev_multiplier, solar, weekend, holiday, traffic_index]
    COND_DIM            = 6

    # ── Model — best-of-class capacity ────────────────────────────────────────
    TCN_CHANNELS        = [128, 256, 256, 256]   # ~4M params, GPU-optimal
    KERNEL_SIZE         = 3
    DROPOUT             = 0.15
    LATENT_DIM          = 128
    DECODER_HIDDEN      = 512

    # ── Training schedule ─────────────────────────────────────────────────────
    EPOCHS              = 200
    BATCH_SIZE          = 64
    LEARNING_RATE       = 2e-4
    GRAD_CLIP_NORM      = 1.0
    LOG_EVERY           = 5
    SAVE_EVERY          = 10

    # KLD annealing schedule (linear warm-up over 50 epochs)
    KLD_WARMUP_EPOCHS   = 50
    KLD_MAX             = 0.10

    # Physics penalty weights — Phase 1 (0) → Phase 2 (activated at epoch 51)
    PHYSICS_EPOCH_ON    = 51      # epoch at which LinDistFlow penalties activate
    LAMBDA_VOLT         = 1000.0  # voltage violation weight
    LAMBDA_THERMAL      = 500.0   # thermal current violation weight
    LAMBDA_XFMR         = 800.0   # transformer capacity violation weight

    # ── Paths ─────────────────────────────────────────────────────────────────
    CHECKPOINT_DIR      = "output/checkpoints"
    BEST_CKPT           = "output/best_model.pt"
    FINAL_CKPT          = "output/gcvae_final.pt"
    LOG_FILE            = "output/training_log.jsonl"
    METRICS_FILE        = "output/metrics_history.json"

# ── Redirect checkpoint paths to Google Drive if on Colab ────────────────────
if DRIVE_DIR:
    CFG.CHECKPOINT_DIR = os.path.join(DRIVE_DIR, "checkpoints")
    CFG.BEST_CKPT      = os.path.join(DRIVE_DIR, "best_model.pt")
    CFG.FINAL_CKPT     = os.path.join(DRIVE_DIR, "gcvae_final.pt")
    CFG.LOG_FILE       = os.path.join(DRIVE_DIR, "training_log.jsonl")
    CFG.METRICS_FILE   = os.path.join(DRIVE_DIR, "metrics_history.json")
    print(f"\n📂  Colab detected — all checkpoints → Google Drive:")
    print(f"    {CFG.CHECKPOINT_DIR}")
else:
    print(f"\n💻  Local mode — checkpoints → output/")

os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)

print(f"\n{'='*55}")
print(f"  EVolvAI — GCD-VAE Configuration")
print(f"{'='*55}")
print(f"  City          : {CFG.CITY}")
print(f"  Data range    : {CFG.DATA_START} → {CFG.DATA_END}")
print(f"  Input dim     : [{CFG.NUM_FEATURES}, {CFG.SEQ_LEN}]  (features × hours)")
print(f"  Condition dim : {CFG.COND_DIM}")
print(f"  Latent dim    : {CFG.LATENT_DIM}")
print(f"  TCN channels  : {CFG.TCN_CHANNELS}")
print(f"  Batch size    : {CFG.BATCH_SIZE}")
print(f"  Max epochs    : {CFG.EPOCHS}")
print(f"  Physics on @  : epoch {CFG.PHYSICS_EPOCH_ON}")
print(f"  λ_volt        : {CFG.LAMBDA_VOLT}")
print(f"  λ_thermal     : {CFG.LAMBDA_THERMAL}")
print(f"  λ_xfmr        : {CFG.LAMBDA_XFMR}")
print(f"{'='*55}")


## §7 · PyTorch Dataset & DataLoader
Fuses the bootstrapped EV demand parquet with NYC ATVC traffic + Open-Meteo weather into a unified `[NUM_FEATURES, SEQ_LEN]` tensor per daily sample.

If per-day traffic profiles are available (`traffic_daily_24h.parquet`), each day gets its **actual measured NYC traffic profile**; otherwise falls back to the canonical 24h mean.


In [ ]:
import datetime

def _date_to_condition(date_str: str, traffic_val: float = 0.5) -> list:
    """Build a 6-D condition vector deterministically from a calendar date."""
    try:
        d   = datetime.date.fromisoformat(date_str)
        dow = d.weekday()
        doy = d.timetuple().tm_yday
        # NYC seasonal solar proxy
        solar   = float(np.clip(0.5 + 0.5 * np.sin((doy - 80) * 2 * np.pi / 365), 0.0, 1.0))
        weekend = float(dow >= 5)
    except Exception:
        return [0.0, 1.0, 0.5, 0.0, 0.0, 0.5]
    return [0.0, 1.0, solar, weekend, 0.0, float(traffic_val)]


class EVDemandDataset(Dataset):
    """
    Loads train_data.parquet + NYC weather + NYC ATVC traffic and fuses them into
    (x: [NUM_FEATURES, SEQ_LEN], cond: [COND_DIM]) tuples.

    Data sources merged per day:
      - demand_kw   : 32 node channels  (bootstrapped from NYC PlugNYC sessions)
      - weather     : 4 channels (temp_c, precip_mm, solar_availability, traffic_index)
                       → Open-Meteo NYC hourly data
      - traffic     : from NYC ATVC (per-day profile if available, else canonical mean)
      - cond vector : 6 scalars (date-derived, reproducible)
    """

    def __init__(self):
        print("\n⚙️   Loading EVDemandDataset...")

        df_ev = pd.read_parquet("data/processed/train_data.parquet")
        print(f"  EV demand parquet : {df_ev.shape[0]:,} rows | {df_ev['date'].nunique():,} days")

        pivot = df_ev.pivot_table(
            index=['date', 'hour'], columns='node_id',
            values='demand_kw', aggfunc='sum', fill_value=0.0
        )
        pivot = pivot.sort_index()

        # Expected node columns: node_00 … node_31
        expected_nodes = [f"node_{i:02d}" for i in range(CFG.NUM_NODES)]
        pivot = pivot.reindex(columns=expected_nodes, fill_value=0.0)

        # Load canonical traffic profile (24,)
        if os.path.exists("data/processed/traffic_tensor_24h.npy"):
            traffic_24h = np.load("data/processed/traffic_tensor_24h.npy")
        else:
            traffic_24h = np.zeros(24, dtype=np.float32)

        # Load per-day ATVC profiles if available
        daily_traffic = None
        if os.path.exists("data/processed/traffic_daily_24h.parquet"):
            dt = pd.read_parquet("data/processed/traffic_daily_24h.parquet")
            dt = dt.set_index('date')
            daily_traffic = dt  # (n_dates, 24 columns h00...h23)
            print(f"  Per-day ATVC      : {len(dt):,} dates available")

        # Load weather (Open-Meteo NYC)
        w_files = ["data/processed/weather_features.parquet", "weather_data.parquet"]
        df_w = None
        for wf in w_files:
            if os.path.exists(wf):
                df_w = pd.read_parquet(wf)
                print(f"  Weather source    : {wf}  ({len(df_w):,} rows)")
                break
        if df_w is None:
            print("  ⚠️   No weather parquet found — using zeros")

        all_dates = sorted(pivot.index.get_level_values('date').unique().tolist())
        print(f"  Building day tensors for {len(all_dates):,} unique dates...")

        demand_list, weather_list, cond_list, valid_dates = [], [], [], []

        for date in all_dates:
            try:
                day_ev = pivot.loc[date]             # [24, 32]
                if len(day_ev) != CFG.SEQ_LEN:
                    continue

                demand_arr = day_ev.values.astype(np.float32)   # [24, 32]

                # Traffic profile for this day
                date_str = str(date)
                if daily_traffic is not None and date_str in daily_traffic.index:
                    day_traffic = daily_traffic.loc[date_str].values.astype(np.float32)  # (24,)
                else:
                    day_traffic = traffic_24h  # fall back to canonical mean

                # Weather channels per hour [24, 4]
                w_arr = np.zeros((CFG.SEQ_LEN, CFG.NUM_WEATHER_FEAT), dtype=np.float32)
                if df_w is not None:
                    day_w = df_w[df_w['date'] == date_str]
                    if len(day_w) >= CFG.SEQ_LEN:
                        if 'temperature_c' in day_w.columns:
                            w_arr[:, 0] = day_w['temperature_c'].values[:CFG.SEQ_LEN]
                        if 'precipitation_mm' in day_w.columns:
                            w_arr[:, 1] = day_w['precipitation_mm'].values[:CFG.SEQ_LEN]
                        if 'solar_availability' in day_w.columns:
                            w_arr[:, 2] = day_w['solar_availability'].values[:CFG.SEQ_LEN]
                w_arr[:, 3] = day_traffic   # channel 3 = NYC ATVC traffic index

                demand_list.append(demand_arr)
                weather_list.append(w_arr)
                cond_list.append(_date_to_condition(date_str, float(day_traffic.mean())))
                valid_dates.append(date)

            except Exception:
                continue

        demand_np  = np.stack(demand_list,  axis=0)   # [N, 24, 32]
        weather_np = np.stack(weather_list, axis=0)   # [N, 24, 4]

        # Z-score normalise each channel independently
        def znorm(arr):
            mu  = arr.mean(axis=(0, 1), keepdims=True)
            std = arr.std(axis=(0, 1),  keepdims=True)
            return (arr - mu) / (std + 1e-6)

        demand_n  = znorm(demand_np)
        weather_n = znorm(weather_np)

        # Concatenate → [N, 24, 36]
        fused = np.concatenate([demand_n, weather_n], axis=-1).astype(np.float32)

        self._data  = fused
        self._conds = np.array(cond_list, dtype=np.float32)
        self._dates = valid_dates

        print(f"  ✅  Dataset ready: {len(self):,} samples  shape={self._data.shape}")
        print(f"      Condition dim={self._conds.shape[1]}  Date range: {valid_dates[0]} → {valid_dates[-1]}")

    def __len__(self): return len(self._data)

    def __getitem__(self, idx):
        x    = torch.from_numpy(self._data[idx]).permute(1, 0)  # [36, 24] Chan-first
        cond = torch.from_numpy(self._conds[idx])
        return x, cond


print("✅  EVDemandDataset class defined")


## §8 · GCD-VAE Model Architecture
Best-of-class configuration:
- **CausalConv1d** — strictly causal (no look-ahead via pre-padding)
- **TCNBlock** — dual-conv residual with exponential dilation
- **Multi-Head Self-Attention** — applied to encoder TCN output for long-range temporal dependency
- **Latent Dim 128** — sufficient bandwidth for 32-node spatial complexity
- **Conditioned Decoder** — Z ⊕ C fed through 512-unit FC → TCN decoder


In [ ]:
class CausalConv1d(nn.Conv1d):
    """Strictly causal 1D convolution (no future leakage). Bai et al. 2018."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, **kwargs):
        self._pad = (kernel_size - 1) * dilation
        super().__init__(in_ch, out_ch, kernel_size,
                         padding=self._pad, dilation=dilation, **kwargs)
    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._pad] if self._pad else out


class TCNBlock(nn.Module):
    """Residual TCN block: two CausalConv1d + layer-norm + dropout."""
    def __init__(self, n_in, n_out, kernel_size, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(n_in,  n_out, kernel_size, dilation=dilation),
            nn.LayerNorm([n_out, CFG.SEQ_LEN]),
            nn.GELU(),
            nn.Dropout(dropout),
            CausalConv1d(n_out, n_out, kernel_size, dilation=dilation),
            nn.LayerNorm([n_out, CFG.SEQ_LEN]),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else nn.Identity()

    def forward(self, x):
        return F.gelu(self.net(x) + self.skip(x))


class TemporalConvNet(nn.Module):
    def __init__(self, n_in, channels, kernel_size, dropout):
        super().__init__()
        layers, prev = [], n_in
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(prev, ch, kernel_size, dilation=2**i, dropout=dropout))
            prev = ch
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class GenerativeCounterfactualVAE(nn.Module):
    """
    GCD-VAE: Causal TCN Encoder + Multi-Head Attention + Conditioned TCN Decoder.
    Physics-aware: LinDistFlow penalties backpropagate through the decoder.
    """
    def __init__(self):
        super().__init__()
        tcn_flat = CFG.SEQ_LEN * CFG.TCN_CHANNELS[-1]

        # Encoder
        self.enc_tcn  = TemporalConvNet(CFG.NUM_FEATURES, CFG.TCN_CHANNELS,
                                         CFG.KERNEL_SIZE, CFG.DROPOUT)
        self.attention = nn.MultiheadAttention(embed_dim=CFG.TCN_CHANNELS[-1],
                                               num_heads=8, dropout=CFG.DROPOUT,
                                               batch_first=True)
        self.attn_norm = nn.LayerNorm(CFG.TCN_CHANNELS[-1])
        self.fc_mu     = nn.Linear(tcn_flat, CFG.LATENT_DIM)
        self.fc_logvar = nn.Linear(tcn_flat, CFG.LATENT_DIM)

        # Decoder
        self.dec_fc  = nn.Sequential(
            nn.Linear(CFG.LATENT_DIM + CFG.COND_DIM, CFG.DECODER_HIDDEN),
            nn.GELU(),
            nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.DECODER_HIDDEN, tcn_flat),
            nn.GELU(),
        )
        self.dec_tcn = TemporalConvNet(CFG.TCN_CHANNELS[-1],
                                        [256, 128, CFG.NUM_FEATURES],
                                        CFG.KERNEL_SIZE, CFG.DROPOUT)

    def encode(self, x):
        h = self.enc_tcn(x)                                    # [B, 256, 24]
        h_t = h.permute(0, 2, 1)                               # [B, 24, 256]
        attn_out, _ = self.attention(h_t, h_t, h_t)
        h_t = self.attn_norm(h_t + attn_out)                   # residual + norm
        h_flat = h_t.flatten(start_dim=1)                       # [B, 24*256]
        return self.fc_mu(h_flat), self.fc_logvar(h_flat)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * (0.5 * logvar).exp()

    def decode(self, z, cond):
        h = self.dec_fc(torch.cat([z, cond], dim=-1))          # [B, 24*256]
        h = h.view(-1, CFG.TCN_CHANNELS[-1], CFG.SEQ_LEN)      # [B, 256, 24]
        return self.dec_tcn(h)                                   # [B, 36, 24]

    def forward(self, x, cond):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar), cond), mu, logvar


# ── Parameter count report ────────────────────────────────────────────────────
_test_model = GenerativeCounterfactualVAE()
total_params  = sum(p.numel() for p in _test_model.parameters())
train_params  = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
enc_params    = sum(p.numel() for p in list(_test_model.enc_tcn.parameters()) +
                                        list(_test_model.attention.parameters()) +
                                        list(_test_model.fc_mu.parameters()) +
                                        list(_test_model.fc_logvar.parameters()))
dec_params    = total_params - enc_params

print(f"\n{'='*50}")
print(f"  GCD-VAE Architecture Summary")
print(f"{'='*50}")
print(f"  Total parameters  : {total_params:>10,}")
print(f"  Trainable params  : {train_params:>10,}")
print(f"  Encoder params    : {enc_params:>10,}")
print(f"  Decoder params    : {dec_params:>10,}")
print(f"  Latent dim        : {CFG.LATENT_DIM}")
print(f"  Attention heads   : 8  (embed={CFG.TCN_CHANNELS[-1]})")
print(f"  Receptive field   : {(CFG.KERNEL_SIZE-1)*(2**len(CFG.TCN_CHANNELS)-1)} time-steps")
del _test_model


## §9 · LinDistFlow Physics Penalty Engine
The IEEE 33-Bus V² form LinDistFlow is implemented as a differentiable `nn.Module`.
Gradients propagate backwards through voltage drops, branch currents, and transformer loading — directly into the TCN decoder weights.

**Penalty formula (IEEE-standard):**
```
L_physics = λ_v ⋅ Σ[max(0, v_min−V_i)² + max(0, V_i−v_max)²]
          + λ_t ⋅ Σ[max(0, |I_k|/I_lim − 1)²]
          + λ_x ⋅ max(0, S_xfmr/S_rated − 1)²
```


In [ ]:
from data_pipeline.physics_penalty_engine import physics_penalty_engine
from generative_core.physics_loss import LinDistFlowLoss

# Instantiate the differentiable LinDistFlow module
lindistflow = LinDistFlowLoss(DEVICE)
lindistflow.eval()  # no dropout in physics engine

print("✅  LinDistFlowLoss module ready")
print(f"   Base MVA      : {lindistflow.base_mva} MVA")
print(f"   V bounds      : [{lindistflow.v_min:.2f}, {lindistflow.v_max:.2f}] p.u.")
print(f"   I limit       : {lindistflow.i_lim_pu:.2f} p.u.")
print(f"   Transformer   : {lindistflow.xfmr_kva:.0f} kVA")

# Quick sanity check — zero load should be feasible
dummy_zero = torch.zeros(1, 24, 32, device=DEVICE)
v_p, t_p, x_p = lindistflow(dummy_zero)
print(f"\n   Sanity check (zero load):")
print(f"     Voltage penalty   = {v_p.item():.6f}  (expected: 0)")
print(f"     Thermal penalty   = {t_p.item():.6f}  (expected: 0)")
print(f"     Transformer pen.  = {x_p.item():.6f}  (expected: 0)")
assert v_p.item() < 1e-5, "FAIL: LinDistFlow should return 0 for zero load"
print("   ✅  Sanity PASSED")


## §10 · Training Engine — Full Metric Suite
Every epoch logs: Total Loss, Reconstruction Loss, KLD, Physics Penalty, R², MAE.
Checkpoints every 10 epochs + best model tracked. Full JSONL training log written to disk.


In [ ]:
from generative_core.physics_loss import LinDistFlowLoss

def compute_r2_mae(targets: np.ndarray, preds: np.ndarray):
    t_flat = targets.reshape(-1)
    p_flat = preds.reshape(-1)
    r2  = float(r2_score(t_flat, p_flat))
    mae = float(mean_absolute_error(t_flat, p_flat))
    return r2, mae


def train_evolvai():
    # ── Setup ─────────────────────────────────────────────────────────────────
    model     = GenerativeCounterfactualVAE().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-5)
    physics   = LinDistFlowLoss(DEVICE)

    dataset   = EVDemandDataset()
    loader    = DataLoader(dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=(DEVICE.type == 'cuda'),
                           prefetch_factor=2)

    print(f"\n{'='*65}")
    print(f"  EVolvAI GCD-VAE Training")
    print(f"{'='*65}")
    print(f"  Device         : {DEVICE}")
    print(f"  City           : {CFG.CITY}")
    print(f"  Dataset        : {len(dataset):,} samples | {len(loader)} batches/epoch")
    print(f"  Epochs         : {CFG.EPOCHS}")
    print(f"  Physics on @   : epoch {CFG.PHYSICS_EPOCH_ON}")
    print(f"  Checkpoint dir : {CFG.CHECKPOINT_DIR}")
    print(f"{'='*65}\n")

    # ── History buffers ────────────────────────────────────────────────────────
    history = {k: [] for k in
               ['epoch', 'total_loss', 'recon_loss', 'kld_loss', 'physics_loss',
                'r2', 'mae', 'kld_weight', 'lr', 'v_penalty', 't_penalty', 'x_penalty',
                'epoch_time_s']}

    best_r2       = -np.inf
    start_epoch   = 1

    # ── Checkpoint resume ──────────────────────────────────────────────────────
    latest_ckpt = os.path.join(CFG.CHECKPOINT_DIR, "latest.pt")
    if os.path.exists(latest_ckpt):
        print("🔄  Resuming from checkpoint...")
        ck = torch.load(latest_ckpt, map_location=DEVICE)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        scheduler.load_state_dict(ck['scheduler_state_dict'])
        start_epoch = ck['epoch'] + 1
        history     = ck.get('history', history)
        best_r2     = ck.get('best_r2', best_r2)
        print(f"   Resumed from epoch {ck['epoch']}  (best R²={best_r2:.4f})")

    log_fh = open(CFG.LOG_FILE, 'a')

    # ── Training loop ──────────────────────────────────────────────────────────
    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        model.train()
        t0 = time.time()

        # KLD annealing: linear ramp 0 → KLD_MAX over first WARMUP epochs
        kld_w = min(CFG.KLD_MAX, (epoch / CFG.KLD_WARMUP_EPOCHS) * CFG.KLD_MAX)

        # Physics: off during warm-up, on after PHYSICS_EPOCH_ON
        phys_on = epoch >= CFG.PHYSICS_EPOCH_ON

        e_total = e_recon = e_kld = e_phys = 0.0
        e_v_pen = e_t_pen = e_x_pen = 0.0
        all_t, all_p = [], []
        n_batches = 0

        for batch_x, batch_cond in loader:
            x, cond = batch_x.to(DEVICE), batch_cond.to(DEVICE)
            optimizer.zero_grad()

            recon, mu, logvar = model(x, cond)

            # ELBO
            recon_loss = F.mse_loss(recon, x, reduction='mean')
            kld_loss   = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss       = recon_loss + kld_w * kld_loss

            # LinDistFlow physics penalty
            v_pen = t_pen = x_pen = torch.tensor(0.0, device=DEVICE)
            if phys_on:
                # Extract demand channels only [B, 32, 24] → [B, 24, 32]
                ev_power = recon[:, :CFG.NUM_NODES, :].permute(0, 2, 1).contiguous()
                v_pen, t_pen, x_pen = physics(ev_power)
                phys_loss = CFG.LAMBDA_VOLT * v_pen + CFG.LAMBDA_THERMAL * t_pen + CFG.LAMBDA_XFMR * x_pen
                loss = loss + phys_loss
                e_phys  += phys_loss.item()
                e_v_pen += v_pen.item()
                e_t_pen += t_pen.item()
                e_x_pen += x_pen.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            optimizer.step()

            e_total += loss.item()
            e_recon += recon_loss.item()
            e_kld   += kld_loss.item()
            all_t.append(x.detach().cpu().numpy())
            all_p.append(recon.detach().cpu().numpy())
            n_batches += 1

        scheduler.step()

        # ── Epoch metrics ──────────────────────────────────────────────────────
        nb = max(n_batches, 1)
        avg_total = e_total / nb
        avg_recon = e_recon / nb
        avg_kld   = e_kld   / nb
        avg_phys  = e_phys  / nb
        avg_v     = e_v_pen / nb
        avg_t     = e_t_pen / nb
        avg_x     = e_x_pen / nb

        t_np = np.concatenate(all_t).flatten()
        p_np = np.concatenate(all_p).flatten()
        r2, mae = compute_r2_mae(t_np, p_np)
        epoch_time = time.time() - t0
        cur_lr = optimizer.param_groups[0]['lr']

        # Store
        for key, val in zip(history.keys(), [
            epoch, avg_total, avg_recon, avg_kld, avg_phys,
            r2, mae, kld_w, cur_lr, avg_v, avg_t, avg_x, epoch_time
        ]):
            history[key].append(val)

        # ── Logging ────────────────────────────────────────────────────────────
        log_entry = {
            'epoch': epoch, 'total_loss': round(avg_total, 6),
            'recon_loss': round(avg_recon, 6), 'kld_loss': round(avg_kld, 6),
            'physics_loss': round(avg_phys, 6), 'r2': round(r2, 6),
            'mae': round(mae, 6), 'kld_weight': round(kld_w, 6),
            'lr': round(cur_lr, 8), 'physics_on': phys_on,
            'v_penalty': round(avg_v, 8), 't_penalty': round(avg_t, 8),
            'x_penalty': round(avg_x, 8), 'epoch_time_s': round(epoch_time, 2)
        }
        log_fh.write(json.dumps(log_entry) + '\n')
        log_fh.flush()

        if epoch % CFG.LOG_EVERY == 0 or epoch == 1:
            phys_tag = f"| Phys: {avg_phys:.4f}" if phys_on else "| Phys: [WARMUP]"
            print(f"Ep {epoch:>4}/{CFG.EPOCHS} | Loss: {avg_total:.5f} "
                  f"| Recon: {avg_recon:.5f} | KLD: {avg_kld:.5f} "
                  f"{phys_tag} | R²: {r2:+.4f} | MAE: {mae:.4f} "
                  f"| lr: {cur_lr:.2e} | {epoch_time:.1f}s")

        # ── Checkpoint ─────────────────────────────────────────────────────────
        def _save_ckpt(path):
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history':              history,
                'best_r2':              best_r2,
                'cfg': {k: v for k, v in vars(CFG).items() if not k.startswith('_')},
            }, path)

        if epoch % CFG.SAVE_EVERY == 0 or epoch == CFG.EPOCHS:
            _save_ckpt(latest_ckpt)
            _save_ckpt(os.path.join(CFG.CHECKPOINT_DIR, f"epoch_{epoch:04d}.pt"))
            print(f"   💾  Checkpoint saved @ epoch {epoch}")

        if r2 > best_r2:
            best_r2 = r2
            _save_ckpt(CFG.BEST_CKPT)
            print(f"   ⭐  New best R²={best_r2:.4f} — best_model.pt updated")

    log_fh.close()
    _save_ckpt(CFG.FINAL_CKPT)

    # Save full metrics as JSON for reproducibility
    with open(CFG.METRICS_FILE, 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  Training complete!")
    print(f"  Best R²  : {best_r2:.4f}")
    print(f"  Final MAE: {history['mae'][-1]:.4f}")
    print(f"  Log file : {CFG.LOG_FILE}")
    print(f"  Metrics  : {CFG.METRICS_FILE}")
    print(f"{'='*65}")

    return model, history, dataset


trained_model, history, dataset = train_evolvai()


## §11 · Training Diagnostics & Publication Figures


In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('EVolvAI GCD-VAE — Training Diagnostics (NYC Dataset)', fontsize=16, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.36)

epochs = history['epoch']

# 1. Total Loss
ax = fig.add_subplot(gs[0, 0])
ax.plot(epochs, history['total_loss'], '#e74c3c', lw=2, label='Total')
ax.plot(epochs, history['recon_loss'], '#3498db', lw=1.5, ls='--', label='Recon')
ax.set_title('Loss Decomposition'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.set_yscale('log')

# 2. KLD
ax = fig.add_subplot(gs[0, 1])
ax.plot(epochs, history['kld_loss'],   '#9b59b6', lw=2, label='KLD')
ax.plot(epochs, history['kld_weight'], '#f39c12', lw=1.5, ls=':', label='KLD weight')
ax.axvline(CFG.PHYSICS_EPOCH_ON, color='red', ls='--', alpha=0.5, label=f'Physics ON @{CFG.PHYSICS_EPOCH_ON}')
ax.set_title('KLD & Annealing'); ax.set_xlabel('Epoch')
ax.legend(fontsize=7)

# 3. Physics Penalty
ax = fig.add_subplot(gs[0, 2])
if any(v > 0 for v in history['physics_loss']):
    ax.plot(epochs, history['physics_loss'], '#e74c3c',  lw=2, label='Total physics')
    ax.plot(epochs, history['v_penalty'],    '#3498db',  lw=1.5, ls='--', label='Voltage pen.')
    ax.plot(epochs, history['t_penalty'],    '#2ecc71',  lw=1.5, ls='--', label='Thermal pen.')
    ax.plot(epochs, history['x_penalty'],    '#f39c12',  lw=1.5, ls='--', label='Xfmr pen.')
else:
    ax.text(0.5, 0.5, 'Physics warm-up\n(activates @ epoch {})'.format(CFG.PHYSICS_EPOCH_ON),
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
ax.set_title('LinDistFlow Penalties'); ax.set_xlabel('Epoch')
ax.legend(fontsize=7)

# 4. R²
ax = fig.add_subplot(gs[1, 0])
ax.plot(epochs, history['r2'], '#27ae60', lw=2)
ax.axhline(0, color='red', ls='--', alpha=0.5)
ax.axhline(0.9, color='green', ls=':', alpha=0.5, label='R²=0.9 target')
ax.fill_between(epochs, history['r2'], alpha=0.15, color='#27ae60')
ax.set_title('Reconstruction R²'); ax.set_xlabel('Epoch'); ax.set_ylabel('R²')
ax.legend()

# 5. MAE
ax = fig.add_subplot(gs[1, 1])
ax.plot(epochs, history['mae'], '#e67e22', lw=2)
ax.fill_between(epochs, history['mae'], alpha=0.15, color='#e67e22')
ax.set_title('Mean Absolute Error'); ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (z-score units)')

# 6. Learning Rate
ax = fig.add_subplot(gs[1, 2])
ax.semilogy(epochs, history['lr'], '#8e44ad', lw=2)
ax.set_title('Learning Rate (Cosine Decay)'); ax.set_xlabel('Epoch'); ax.set_ylabel('LR')

# 7. Reconstruction sample (last batch)
ax = fig.add_subplot(gs[2, :2])
trained_model.eval()
loader_s = DataLoader(dataset, batch_size=8, shuffle=False)
x_s, c_s = next(iter(loader_s))
with torch.no_grad():
    r_s, _, _ = trained_model(x_s.to(DEVICE), c_s.to(DEVICE))
true_node0 = x_s[:, 0, :].numpy()   # [8, 24]
pred_node0 = r_s[:, 0, :].cpu().numpy()
for k in range(min(4, len(true_node0))):
    ax.plot(true_node0[k], alpha=0.6, lw=1.5, label='True' if k == 0 else '')
    ax.plot(pred_node0[k], alpha=0.6, lw=1.5, ls='--', label='Recon' if k == 0 else '')
ax.set_title('Sample Reconstructions — Node 0 (z-score)'); ax.set_xlabel('Hour'); ax.legend()

# 8. Epoch time
ax = fig.add_subplot(gs[2, 2])
ax.bar(epochs, history['epoch_time_s'], color='#7f8c8d', alpha=0.7)
ax.set_title('Training Time / Epoch'); ax.set_xlabel('Epoch'); ax.set_ylabel('Seconds')

plt.savefig('output/training_diagnostics.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅  Figure saved → output/training_diagnostics.png")


## §12 · Final Metrics Report (Paper-Ready)

In [ ]:
# Load best checkpoint and run evaluation
best_model = GenerativeCounterfactualVAE().to(DEVICE)
ck = torch.load(CFG.BEST_CKPT, map_location=DEVICE)
best_model.load_state_dict(ck['model_state_dict'])
best_model.eval()

# Full pass over dataset
eval_loader = DataLoader(dataset, batch_size=128, shuffle=False)
all_true, all_pred = [], []
with torch.no_grad():
    for bx, bc in eval_loader:
        x, c = bx.to(DEVICE), bc.to(DEVICE)
        recon, mu, logvar = best_model(x, c)
        all_true.append(x.cpu().numpy())
        all_pred.append(recon.cpu().numpy())

true_np = np.concatenate(all_true)   # [N, 36, 24]
pred_np = np.concatenate(all_pred)

# Report
r2_final  = r2_score(true_np.flatten(),  pred_np.flatten())
mae_final = mean_absolute_error(true_np.flatten(), pred_np.flatten())
zero_pct  = np.mean(np.abs(pred_np[:, :CFG.NUM_NODES, :]) < 1e-4) * 100

total_params = sum(p.numel() for p in best_model.parameters())

print("=" * 60)
print("  EVolvAI GCD-VAE  —  Final Evaluation Report")
print("=" * 60)
print(f"  City                    : {CFG.CITY}")
print(f"  Data Range              : {CFG.DATA_START} → {CFG.DATA_END}")
print(f"  Model Parameters        : {total_params:,} (~{total_params/1e6:.1f}M)")
print(f"  Training Samples        : {len(dataset):,}  (5,000 days × 32 nodes)")
print(f"  Final R²                : {r2_final:+.4f}")
print(f"  Final MAE               : {mae_final:.4f} (z-score units)")
print(f"  Best Epoch R²           : {max(history['r2']):.4f} @ ep {history['epoch'][history['r2'].index(max(history['r2']))]}")
print(f"  Zero-output %           : {zero_pct:.2f}%  (target < 2%)")
print(f"  Physics active          : epoch {CFG.PHYSICS_EPOCH_ON}+")
print(f"  λ_volt / λ_thermal / λ_xfmr : {CFG.LAMBDA_VOLT} / {CFG.LAMBDA_THERMAL} / {CFG.LAMBDA_XFMR}")
print(f"  LinDistFlow V bounds    : [{lindistflow.v_min:.2f}, {lindistflow.v_max:.2f}] p.u.")
print(f"  Best checkpoint         : {CFG.BEST_CKPT}")
print(f"  Training log            : {CFG.LOG_FILE}")
print("=" * 60)

# Machine-readable summary
summary = {
    'city': CFG.CITY, 'data_start': CFG.DATA_START, 'data_end': CFG.DATA_END,
    'total_params': total_params, 'r2_final': round(r2_final, 6),
    'mae_final': round(mae_final, 6), 'zero_output_pct': round(zero_pct, 4),
    'best_r2': round(max(history['r2']), 6),
    'best_epoch': history['epoch'][history['r2'].index(max(history['r2']))],
    'dataset_samples': len(dataset), 'num_epochs_run': len(history['epoch']),
}
with open('output/eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("\n✅  Evaluation summary saved → output/eval_summary.json")


## §13 · Counterfactual Demand Generation
Generate 5 physically-bounded extreme NYC scenarios by injecting different condition vectors C into the trained decoder with a fixed latent Z.

Scenarios are grounded in NYC-specific conditions: winter storms, summer EV peaks, rush-hour gridlock on major NYC arteries.


In [ ]:
SCENARIOS = {
    "baseline"             : [0.0,  1.0, 1.0, 0.0, 0.0, 0.50],
    "nyc_winter_storm"     : [1.0,  2.5, 0.0, 0.0, 0.0, 0.85],   # Jan blizzard, high EV demand
    "full_electrification" : [0.0,  3.0, 1.0, 0.0, 0.0, 0.65],   # 100% EV fleet
    "nyc_rush_hour"        : [0.0,  2.0, 0.5, 0.0, 0.0, 1.00],   # peak ATVC traffic
    "nyc_summer_ev_peak"   : [0.5,  1.5, 1.0, 0.0, 0.0, 0.90],   # Jul/Aug heatwave
}

os.makedirs("output", exist_ok=True)
best_model.eval()

# Fix a single latent draw Z from the dataset mean
with torch.no_grad():
    sample_x, _ = next(iter(DataLoader(dataset, batch_size=64, shuffle=True)))
    mu_z, logvar_z = best_model.encode(sample_x.to(DEVICE))
    Z_fixed = mu_z.mean(dim=0, keepdim=True)   # [1, 128]

print("📊  Generating NYC counterfactual scenarios...")
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(22, 4), sharey=False)

for ax, (name, cond_vals) in zip(axes, SCENARIOS.items()):
    cond_t = torch.tensor([cond_vals], dtype=torch.float32, device=DEVICE)  # [1, 6]
    with torch.no_grad():
        gen_out = best_model.decode(Z_fixed, cond_t)   # [1, 36, 24]
    
    # Demand channels only: [32, 24] → mean over nodes → [24]
    demand = gen_out[0, :CFG.NUM_NODES, :].cpu().numpy()   # [32, 24]
    mean_demand = demand.mean(axis=0)                       # [24]
    peak_demand = demand.max(axis=0)                        # [24]
    
    # Save
    np.save(f"output/scenario_{name}.npy", demand)
    
    ax.fill_between(range(24), mean_demand, alpha=0.35, label='Mean node demand')
    ax.plot(range(24), peak_demand, lw=1.5, ls='--', label='Peak node demand')
    ax.plot(range(24), mean_demand, lw=2)
    ax.set_title(name.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.set_xlabel('Hour of Day'); ax.set_ylabel('kW (z-score)') if name == 'baseline' else None
    ax.legend(fontsize=6)
    print(f"  ✅  {name:30s}  peak={peak_demand.max():.3f}  mean={mean_demand.mean():.3f}")

plt.suptitle('EVolvAI NYC Counterfactual Demand Scenarios (Fixed Z, Varying C)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/counterfactual_scenarios.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n✅  Scenarios saved → output/scenario_*.npy")
print("✅  Figure saved   → output/counterfactual_scenarios.png")
